In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

c:\Users\roger\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [5]:
class ZedBoardMNIST(nn.Module):
    def __init__(self):
        super(ZedBoardMNIST, self).__init__()
        # FPGA PL: 1 input channel -> 8 output filters (3x3 kernel, bias=False)
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=0, bias=False)
        
        # ARM PS: MaxPool2d + Fully Connected
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # 26x26 -> 13x13
        self.fc1 = nn.Linear(8 * 13 * 13, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
FRACTIONAL_BITS = 8
SCALE = 1 << FRACTIONAL_BITS  # 256 for Q8.8

def quantize_to_int16(tensor_np):
    """Scales float32 to Q8.8 fixed-point and clamps to signed int16 range."""
    scaled = np.round(tensor_np * SCALE)
    clamped = np.clip(scaled, -32768, 32767)
    return clamped.astype(np.int16)

def export_int16_weights_complete(model, test_loader, device, export_dir="zedboard_int16"):
    os.makedirs(export_dir, exist_ok=True)
    model.eval()

    # 1. Extract floating-point parameters from PyTorch
    conv1_w_f = model.conv1.weight.detach().cpu().numpy()  # (8, 1, 3, 3)
    fc1_w_f   = model.fc1.weight.detach().cpu().numpy()    # (32, 1352)
    fc1_b_f   = model.fc1.bias.detach().cpu().numpy()      # (32,)
    fc2_w_f   = model.fc2.weight.detach().cpu().numpy()    # (10, 32)
    fc2_b_f   = model.fc2.bias.detach().cpu().numpy()      # (10,)

    # 2. Quantize parameters to signed int16 (Q8.8 format)
    conv1_w_q = quantize_to_int16(conv1_w_f)
    fc1_w_q   = quantize_to_int16(fc1_w_f)
    fc1_b_q   = quantize_to_int16(fc1_b_f)
    fc2_w_q   = quantize_to_int16(fc2_w_f)
    fc2_b_q   = quantize_to_int16(fc2_b_f)

    # 3. Fixed-Point Verification Pass
    print("\nRunning fixed-point Q8.8 emulation check on test set...")
    correct = 0
    total = 0
    w_conv_deq = torch.tensor(conv1_w_q.astype(np.float32) / SCALE)
    fc1_w_deq  = fc1_w_q.astype(np.float32) / SCALE
    fc1_b_deq  = fc1_b_q.astype(np.float32) / SCALE
    fc2_w_deq  = fc2_w_q.astype(np.float32) / SCALE
    fc2_b_deq  = fc2_b_q.astype(np.float32) / SCALE

    with torch.no_grad():
        for data, target in test_loader:
            x_conv = F.conv2d(data, w_conv_deq, padding=0)
            x_conv = F.relu(x_conv)
            x_pool = F.max_pool2d(x_conv, kernel_size=2, stride=2)
            
            flat = x_pool.view(x_pool.size(0), -1).numpy()
            fc1_out = np.maximum(0, flat @ fc1_w_deq.T + fc1_b_deq)
            logits  = fc1_out @ fc2_w_deq.T + fc2_b_deq
            
            preds = np.argmax(logits, axis=1)
            correct += int(np.sum(preds == target.numpy()))
            total += len(target)

    print(f"Fixed-Point (Q8.8) Emulated Accuracy: {100.0 * correct / total:.2f}%")

    # 4. Export Raw Binary Files (.bin)
    conv1_w_q.tofile(os.path.join(export_dir, "conv1_w_int16.bin"))
    fc1_w_q.tofile(os.path.join(export_dir, "fc1_w_int16.bin"))
    fc1_b_q.tofile(os.path.join(export_dir, "fc1_b_int16.bin"))
    fc2_w_q.tofile(os.path.join(export_dir, "fc2_w_int16.bin"))
    fc2_b_q.tofile(os.path.join(export_dir, "fc2_b_int16.bin"))

    # 5. Export Complete C Header (weights_int16.h)
    c_header_path = os.path.join(export_dir, "weights_int16.h")
    with open(c_header_path, "w") as f:
        f.write("#ifndef WEIGHTS_INT16_H\n#define WEIGHTS_INT16_H\n\n#include <stdint.h>\n\n")
        f.write(f"#define FIXED_POINT_SHIFT {FRACTIONAL_BITS}\n\n")

        # Conv1: [8][9]
        f.write("// Conv1 Weights: [8 filters][9 kernel values] (Q8.8 format)\n")
        f.write("static const int16_t CONV1_WEIGHTS[8][9] = {\n")
        for kernel in conv1_w_q.reshape(8, 9).tolist():
            f.write("    { " + ", ".join(f"{int(v):6d}" for v in kernel) + " },\n")
        f.write("};\n\n")

        # FC1 Weights: [32][1352]
        f.write("// FC1 Weights: [32 output units][1352 input units]\n")
        f.write("static const int16_t FC1_WEIGHTS[32][1352] = {\n")
        for row in fc1_w_q.tolist():
            f.write("    { " + ", ".join(f"{int(v):6d}" for v in row) + " },\n")
        f.write("};\n\n")

        # FC1 Bias: [32]
        f.write("// FC1 Bias: [32 output units]\n")
        f.write("static const int16_t FC1_BIAS[32] = {\n")
        f.write("    " + ", ".join(f"{int(v):6d}" for v in fc1_b_q.tolist()) + "\n};\n\n")

        # FC2 Weights: [10][32]
        f.write("// FC2 Weights: [10 output units][32 input units]\n")
        f.write("static const int16_t FC2_WEIGHTS[10][32] = {\n")
        for row in fc2_w_q.tolist():
            f.write("    { " + ", ".join(f"{int(v):6d}" for v in row) + " },\n")
        f.write("};\n\n")

        # FC2 Bias: [10]
        f.write("// FC2 Bias: [10 output units]\n")
        f.write("static const int16_t FC2_BIAS[10] = {\n")
        f.write("    " + ", ".join(f"{int(v):6d}" for v in fc2_b_q.tolist()) + "\n};\n\n")

        f.write("#endif // WEIGHTS_INT16_H\n")

    # 6. Export Verilog Hex Initialization File (conv1_weights.hex)
    hex_path = os.path.join(export_dir, "conv1_weights.hex")
    with open(hex_path, "w") as f:
        for val in conv1_w_q.flatten().tolist():
            f.write(f"{int(val) & 0xFFFF:04X}\n")

    print(f"\nAll files successfully exported to '{export_dir}/':")
    print(f"  - C Header: {c_header_path}")
    print(f"  - Verilog HEX: {hex_path}")
    print(f"  - Binary buffers: conv1_w_int16.bin, fc1_w_int16.bin, etc.")

In [12]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    transform = transforms.Compose([transforms.ToTensor()])

    train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
    test_set  = datasets.MNIST("./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_set, batch_size=1000, shuffle=False)

    model = ZedBoardMNIST().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, 6):
        model.train()
        running_loss = 0.0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                pred = model(data).argmax(dim=1)
                correct += pred.eq(target).sum().item()

        print(f"Epoch {epoch}/5 - Loss: {running_loss/len(train_loader):.4f} - Float Test Acc: {100.0 * correct / len(test_set):.2f}%")

    export_int16_weights_complete(model, test_loader, device)

if __name__ == "__main__":
    main()

Training on: cuda
Epoch 1/5 - Loss: 0.3414 - Float Test Acc: 95.41%
Epoch 2/5 - Loss: 0.1275 - Float Test Acc: 97.02%
Epoch 3/5 - Loss: 0.0909 - Float Test Acc: 97.59%
Epoch 4/5 - Loss: 0.0723 - Float Test Acc: 97.61%
Epoch 5/5 - Loss: 0.0599 - Float Test Acc: 97.79%


TypeError: export_int16_weights_complete() takes from 1 to 2 positional arguments but 3 were given